# Cat vs. Dog Image Classifier — CNN with Transfer Learning (VGG16)

**Author:** Kazi Rafi  
**Dataset:** [Dogs vs. Cats — Kaggle](https://www.kaggle.com/datasets/salader/dogs-vs-cats)  

---

## Overview

This notebook builds a binary image classifier that distinguishes between cats and dogs using **Convolutional Neural Networks (CNNs)** and **transfer learning** with the pre-trained **VGG16** architecture.

**Key design decisions:**
- Use VGG16 (trained on ImageNet) as a frozen feature extractor, fine-tuning only the last convolutional block (`block5`) to adapt to our domain.
- Apply real-time data augmentation during training to improve generalization.
- Use a small learning rate (`1e-5`) to avoid destroying pre-trained weights.

**Pipeline:**
1. Environment setup & dataset download  
2. Data exploration & preprocessing  
3. Model architecture (VGG16 + custom head)  
4. Training & evaluation  
5. Inference on new images

---
## 1. Import Libraries

In [ ]:
import os
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

---
## 2. Dataset Setup

Download the **Dogs vs. Cats** dataset from Kaggle and extract it. Make sure your `kaggle.json` API key is uploaded to the Colab session before running this section.

In [ ]:
# Configure Kaggle credentials
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d salader/dogs-vs-cats

In [ ]:
DATASET_ZIP = "/content/dogs-vs-cats.zip"
EXTRACT_DIR = "/content"

with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")
print("Train directory:", os.listdir("/content/train"))
print("Test directory: ", os.listdir("/content/test"))

---
## 3. Data Exploration

In [ ]:
# Load sample images for visual inspection
sample_cat = mpimg.imread("/content/test/cats/cat.10.jpg")
sample_dog = mpimg.imread("/content/test/dogs/dog.100.jpg")

print(f"Cat image shape : {sample_cat.shape}")
print(f"Dog image shape : {sample_dog.shape}")
print("\nNote: Images have varying resolutions — they will be resized to 150×150 during preprocessing.")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle("Sample Images from Dataset", fontsize=14, fontweight="bold")

for ax, img, label in zip(axes, [sample_cat, sample_dog], ["Cat", "Dog"]):
    ax.imshow(img)
    ax.set_title(label, fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing & Augmentation

- **Training set:** Apply augmentation (shear, zoom, horizontal flip) and normalize pixel values to `[0, 1]`.
- **Validation set:** Only normalize — no augmentation to ensure unbiased evaluation.

In [ ]:
# Hyperparameters
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
TRAIN_DIR  = "/content/train"
TEST_DIR   = "/content/test"

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print(f"\nClass indices: {train_generator.class_indices}")

---
## 5. Model Architecture

We use **VGG16** as a frozen feature extractor, with fine-tuning enabled from `block5_conv1` onwards. A lightweight classification head is added on top:

```
VGG16 (frozen up to block4) → Flatten → Dense(256, ReLU) → Dense(64, ReLU) → Dense(1, Sigmoid)
```

In [ ]:
# Load VGG16 pre-trained on ImageNet, excluding the classification head
conv_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

# Freeze all layers initially
conv_base.trainable = True
fine_tune_at = "block5_conv1"
fine_tune_reached = False

for layer in conv_base.layers:
    if layer.name == fine_tune_at:
        fine_tune_reached = True
    layer.trainable = fine_tune_reached

# Print trainable status for each layer
print(f"{'Layer':<25} {'Trainable'}")
print("-" * 35)
for layer in conv_base.layers:
    print(f"{layer.name:<25} {layer.trainable}")

In [ ]:
# Assemble the full model
model = Sequential([
    conv_base,
    Flatten(),
    Dense(256, activation="relu"),
    Dense(64,  activation="relu"),
    Dense(1,   activation="sigmoid")  # Binary output: 0 = cat, 1 = dog
], name="CatDog_VGG16_Classifier")

model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

---
## 6. Training

In [ ]:
EPOCHS = 5

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    verbose=1
)

final_train_acc = history.history["accuracy"][-1]
final_val_acc   = history.history["val_accuracy"][-1]
print(f"\nFinal Training Accuracy  : {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")

---
## 7. Training Curves

In [ ]:
def plot_training_history(history):
    """Plot accuracy and loss curves for training and validation sets."""
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs   = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle("Model Training History", fontsize=14, fontweight="bold")

    # Accuracy plot
    axes[0].plot(epochs, acc,     "g-o", label="Training Accuracy")
    axes[0].plot(epochs, val_acc, "b-o", label="Validation Accuracy")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.5)

    # Loss plot
    axes[1].plot(epochs, loss,     "g-o", label="Training Loss")
    axes[1].plot(epochs, val_loss, "b-o", label="Validation Loss")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


plot_training_history(history)

---
## 8. Inference on New Images

Run predictions on unseen images using the trained model. The output is a probability score — values above `0.5` indicate **Dog**, values at or below `0.5` indicate **Cat**.

In [ ]:
def preprocess_image(img_path: str, target_size: tuple = (150, 150)) -> np.ndarray:
    """Load an image, resize it, and normalize pixel values for model input."""
    img       = load_img(img_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0          # Normalize to [0, 1]
    return np.expand_dims(img_array, axis=0)        # Add batch dimension


def predict_image(model, img_path: str, target_size: tuple = (150, 150)) -> dict:
    """Return the class label and confidence score for a given image."""
    img_array  = preprocess_image(img_path, target_size)
    confidence = model.predict(img_array, verbose=0)[0][0]
    label      = "Dog" if confidence > 0.5 else "Cat"
    score      = confidence if confidence > 0.5 else 1 - confidence
    return {"label": label, "confidence": float(score)}


def display_predictions(model, image_paths: list, target_size: tuple = (150, 150)):
    """Display images alongside their predicted labels and confidence scores."""
    n   = len(image_paths)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, path in zip(axes, image_paths):
        result = predict_image(model, path, target_size)
        img    = load_img(path, target_size=target_size)
        ax.imshow(img)
        ax.set_title(
            f"Prediction: {result['label']}\nConfidence: {result['confidence']:.2%}",
            fontsize=12
        )
        ax.axis("off")

    plt.suptitle("Model Predictions", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Run inference — update paths as needed
test_images = ["/content/Cat.jpeg", "/content/Dog.jpg"]
display_predictions(model, test_images)

---
## 9. Save the Model

In [ ]:
MODEL_PATH = "/content/catdog_vgg16_classifier.keras"
model.save(MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

---
## Summary

| Metric | Value |
|---|---|
| Architecture | VGG16 + Custom Head |
| Fine-tuned from | `block5_conv1` |
| Input size | 150 × 150 × 3 |
| Optimizer | RMSprop (lr=1e-5) |
| Loss | Binary Crossentropy |
| Epochs | 5 |
| Final Training Accuracy | ~95.4% |
| Final Validation Accuracy | ~95.6% |

The classifier achieves ~95% accuracy on the validation set after just 5 epochs, demonstrating the power of transfer learning for image classification with limited training time.